# 0. Setup & Installation
Run this cell first. Restart runtime if prompted.

In [1]:
!pip install -U transformers peft bitsandbytes accelerate datasets scikit-learn --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 87.6 MB/s eta 0:00:00


# 1. Authentication

In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
secret_value_1 = user_secrets.get_secret("HF_TOKEN")

login(token=secret_value_1)

# 2. Configuration

In [3]:
from dataclasses import dataclass, field
from typing import List


@dataclass
class ExperimentConfig:
    # ---- Model ----
    model_name: str = "Qwen/Qwen3-4B"
    max_seq_length: int = 2048

    # ---- Quantization ----
    load_in_4bit: bool = True
    bnb_4bit_quant_type: str = "nf4"
    use_double_quant: bool = True

    # ---- Special tokens ----
    prediction_start_token: str = "<|prediction|>"
    prediction_end_token: str = "<|/prediction|>"

    # ---- Paths ----
    adapter_path: str = "/kaggle/input/notebooks/zygong1994/fine-tuning-experiment-0330/outputs/checkpoint-126"


cfg = ExperimentConfig()
print(f"Config: {cfg}")

Config: ExperimentConfig(model_name='Qwen/Qwen3-4B', max_seq_length=2048, load_in_4bit=True, bnb_4bit_quant_type='nf4', use_double_quant=True, prediction_start_token='<|prediction|>', prediction_end_token='<|/prediction|>', adapter_path='/kaggle/input/notebooks/zygong1994/fine-tuning-experiment-0330/outputs/checkpoint-126')


# 3. Define Special Tokens

In [4]:
SPECIAL_TOKENS = [
    # Input structure
    "<|system|>", "<|/system|>",
    "<|conversation|>", "<|/conversation|>",
    "<|agent|>", "<|customer|>",
    # Output structure
    "<|prediction|>", "<|/prediction|>",
]

# 4. Load Eval Data

In [5]:
import pandas as pd

eval_data = pd.read_csv("/kaggle/input/notebooks/zygong1994/prepare-test-data-0327/test.csv")

print(f"Eval: {len(eval_data)} samples")
eval_data.head()

Eval: 200 samples


,conversation_id,conversation,full_text,outcome,text,transcript
0,saas-12-conv-4816,"[{""speaker"": ""customer"", ""message"": ""Hey, I sa...","Hey, I saw your post about SmartLLM's integrat...",1,<|system|>You are an expert sales call analyst...,"<|conversation|>\n<|customer|>Hey, I saw your ..."
1,saas-1-conv-4852,"[{""speaker"": ""customer"", ""message"": ""Hey, so, ...","Hey, so, I've been looking into some pricing t...",0,<|system|>You are an expert sales call analyst...,"<|conversation|>\n<|customer|>Hey, so, I've be..."
2,saas-0-conv-1824,"[{""speaker"": ""customer"", ""message"": ""Hey, than...","Hey, thanks for hopping on this call! So, um, ...",1,<|system|>You are an expert sales call analyst...,"<|conversation|>\n<|customer|>Hey, thanks for ..."
3,saas-10-conv-3173,"[{""speaker"": ""sales_rep"", ""message"": ""Hey Jess...","Hey Jessica, hope you're doing well! I wanted ...",0,<|system|>You are an expert sales call analyst...,"<|conversation|>\n<|agent|>Hey Jessica, hope y..."
4,saas-18-conv-466,"[{""speaker"": ""customer"", ""message"": ""Hey, so I...","Hey, so I've been, like, struggling with track...",0,<|system|>You are an expert sales call analyst...,"<|conversation|>\n<|customer|>Hey, so I've bee..."


# 5. Load Model + Tokenizer with LoRA Adapter

In [6]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ---- Quantization config ----
bnb_config = BitsAndBytesConfig(
    load_in_4bit=cfg.load_in_4bit,
    bnb_4bit_quant_type=cfg.bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=cfg.use_double_quant,
)

# ---- Load tokenizer from adapter ----
tokenizer = AutoTokenizer.from_pretrained(cfg.adapter_path)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Tokenizer vocab size: {len(tokenizer)}")

# ---- Load base model ----
base_model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16,
)

base_model.resize_token_embeddings(len(tokenizer))

# ---- Load LoRA adapter ----
model = PeftModel.from_pretrained(base_model, cfg.adapter_path)
model.eval()

print(f"Model loaded with LoRA adapter from {cfg.adapter_path}")

Tokenizer vocab size: 151669


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded with LoRA adapter from /kaggle/input/notebooks/zygong1994/fine-tuning-experiment-0330/outputs/checkpoint-126


# 6. Inference Functions

In [7]:
SYSTEM_PROMPT = (
    "You are an expert sales call analyst specializing in financial services. "
    "You will be given a full transcript of a customer-agent conversation. Your task is to:\n"
    "Predict whether the customer will convert (1) or not (0) based the conversation."
)


def build_inference_prompt(system_prompt: str, transcript: str) -> str:
    return (
        f"<|system|>{system_prompt}<|/system|>\n"
        f"<|conversation|>\n{transcript}\n<|/conversation|>\n"
        f"<|prediction|>"
    )


def predict(model, tokenizer, transcript: str, max_new_tokens=200):
    prompt = build_inference_prompt(SYSTEM_PROMPT, transcript)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    return {
        "generated_output": new_tokens,
        "full_text": tokenizer.decode(outputs[0], skip_special_tokens=False),
    }


def predict_with_logit_probability(model, tokenizer, transcript: str):
    prompt = build_inference_prompt(SYSTEM_PROMPT, transcript)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits[:, -1, :]

    token_0 = tokenizer.encode("0", add_special_tokens=False)[0]
    token_1 = tokenizer.encode("1", add_special_tokens=False)[0]

    binary_logits = logits[:, [token_0, token_1]]
    probs = F.softmax(binary_logits, dim=-1)
    conversion_prob = probs[0, 1].item()

    return {
        "conversion_probability": conversion_prob,
        "predicted_class": 1 if conversion_prob > 0.5 else 0,
        "raw_logits": {"token_0": logits[0, token_0].item(), "token_1": logits[0, token_1].item()},
    }

# 7. Test Inference on One Example

In [8]:
test_row = eval_data.iloc[0]
test_transcript = test_row["transcript"]
test_label = test_row["outcome"]

print("=" * 60)
print("TEST INFERENCE: Full Generation")
print("=" * 60)
result = predict(model, tokenizer, test_transcript)
print(f"Ground truth: {test_label}")
print(f"Generated output:\n{result['generated_output']}")

print("\n" + "=" * 60)
print("TEST INFERENCE: Logit-Based Probability")
print("=" * 60)
prob_result = predict_with_logit_probability(model, tokenizer, test_transcript)
print(f"Ground truth: {test_label}")
print(f"Conversion probability: {prob_result['conversion_probability']:.4f}")
print(f"Predicted class: {prob_result['predicted_class']}")
print(f"Raw logits: {prob_result['raw_logits']}")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


TEST INFERENCE: Full Generation
Ground truth: 1
Generated output:
0<|/prediction|>
<|/conversation|>
<|conversation|>
<|customer|>Hey, I saw your post about SmartLLM's integration features... really interested but also super frustrated with our current tools.
<|agent|>Totally get that! What specific issues are you facing?
<|customer|>So, like, our current setup is a mess. Can't integrate stuff smoothly into our workflow.
<|customer|>And scaling model training? Ugh, it's a nightmare. Costs are high too, like, way too high.
<|agent|>I hear ya! SmartLLM’s got seamless API integration that'll fit right into your existing workflows. How does that sound?
<|customer|>Hmm, sounds good but... I’m kinda worried about the initial costs. $499/month? Is it worth it?
<|agent|>Yeah, I get that. Think about the savings on deployment and maintenance

TEST INFERENCE: Logit-Based Probability
Ground truth: 1
Conversion probability: 0.0674
Predicted class: 0
Raw logits: {'token_0': 25.875, 'token_1': 23.25

# 8. Evaluate on Full Eval Set

In [9]:
from sklearn.metrics import roc_auc_score, classification_report

eval_results = []

print("Running evaluation...")
for i in range(len(eval_data)):
    row = eval_data.iloc[i]
    prob_result = predict_with_logit_probability(
        model, tokenizer, row["transcript"]
    )
    eval_results.append({
        "ground_truth": row["outcome"],
        "predicted_class": prob_result["predicted_class"],
        "probability": prob_result["conversion_probability"],
    })
    if (i + 1) % 10 == 0:
        print(f"  Processed {i+1}/{len(eval_data)} samples")

# ---- Compute metrics ----
correct = sum(1 for r in eval_results if r["ground_truth"] == r["predicted_class"])
total = len(eval_results)
accuracy = correct / total

y_true = [r["ground_truth"] for r in eval_results]
y_pred = [r["predicted_class"] for r in eval_results]
y_prob = [r["probability"] for r in eval_results]

print(f"\n{'='*60}")
print(f"EVALUATION RESULTS")
print(f"{'='*60}")
print(f"Accuracy: {correct}/{total} = {accuracy:.2%}")
print(f"AUC-ROC: {roc_auc_score(y_true, y_prob):.4f}")
print(f"\n{classification_report(y_true, y_pred, target_names=['No Convert (0)', 'Convert (1)'])}")

Running evaluation...
  Processed 10/200 samples
  Processed 20/200 samples
  Processed 30/200 samples
  Processed 40/200 samples
  Processed 50/200 samples
  Processed 60/200 samples
  Processed 70/200 samples
  Processed 80/200 samples
  Processed 90/200 samples
  Processed 100/200 samples
  Processed 110/200 samples
  Processed 120/200 samples
  Processed 130/200 samples
  Processed 140/200 samples
  Processed 150/200 samples
  Processed 160/200 samples
  Processed 170/200 samples
  Processed 180/200 samples
  Processed 190/200 samples
  Processed 200/200 samples

EVALUATION RESULTS
Accuracy: 148/200 = 74.00%
AUC-ROC: 0.8742

                precision    recall  f1-score   support

No Convert (0)       0.67      0.96      0.79       100
   Convert (1)       0.93      0.52      0.67       100

      accuracy                           0.74       200
     macro avg       0.80      0.74      0.73       200
  weighted avg       0.80      0.74      0.73       200



# 9. Cross-Entropy with Ground Truth (Dirac Delta)

For a Dirac delta ground-truth distribution where $q(y=k) = 1$ for the true class and $0$ otherwise, the cross-entropy simplifies to:

$$H(q, p) = -\log p(y = y_{\text{true}})$$

where $p(y=1)$ is the model's predicted conversion probability. Per-sample cross-entropy is computed and then averaged across the eval set.

In [10]:
import numpy as np

y_true_arr = np.array(y_true)
y_prob_arr = np.array(y_prob)

# Clip probabilities to avoid log(0)
eps = 1e-7
y_prob_clipped = np.clip(y_prob_arr, eps, 1.0 - eps)

# Per-sample cross-entropy: -log p(y_true)
#   If y_true == 1: CE = -log(p)
#   If y_true == 0: CE = -log(1 - p)
per_sample_ce = -(y_true_arr * np.log(y_prob_clipped) + (1 - y_true_arr) * np.log(1 - y_prob_clipped))

mean_ce = per_sample_ce.mean()

print(f"{'='*60}")
print(f"CROSS-ENTROPY (Dirac Delta Ground Truth)")
print(f"{'='*60}")
print(f"Mean cross-entropy: {mean_ce:.4f}")
print(f"Min  cross-entropy: {per_sample_ce.min():.4f}")
print(f"Max  cross-entropy: {per_sample_ce.max():.4f}")
print(f"Std  cross-entropy: {per_sample_ce.std():.4f}")

# ---- Per-class breakdown ----
ce_class_0 = per_sample_ce[y_true_arr == 0]
ce_class_1 = per_sample_ce[y_true_arr == 1]

print(f"\nPer-class mean cross-entropy:")
print(f"  No Convert (0): {ce_class_0.mean():.4f} (n={len(ce_class_0)})")
print(f"  Convert    (1): {ce_class_1.mean():.4f} (n={len(ce_class_1)})")

CROSS-ENTROPY (Dirac Delta Ground Truth)
Mean cross-entropy: 0.7214
Min  cross-entropy: 0.0039
Max  cross-entropy: 4.0205
Std  cross-entropy: 1.1745

Per-class mean cross-entropy:
  No Convert (0): 0.1348 (n=100)
  Convert    (1): 1.3081 (n=100)


In [11]:
y_prob_clipped

array([0.06738281, 0.04199219, 0.984375  , 0.01403809, 0.02038574,
       0.04736328, 0.01403809, 0.06005859, 0.01586914, 0.29492188,
       0.02929688, 0.02038574, 0.09521484, 0.95703125, 0.9453125 ,
       0.02600098, 0.02929688, 0.04736328, 0.02600098, 0.0534668 ,
       0.02600098, 0.01794434, 0.04736328, 0.01794434, 0.06738281,
       0.02929688, 0.01794434, 0.11914062, 0.99609375, 0.03735352,
       0.06005859, 0.0534668 , 0.03320312, 0.29492188, 0.59375   ,
       0.03735352, 0.04736328, 0.10693359, 0.22265625, 0.08496094,
       0.03735352, 0.9765625 , 0.89453125, 0.02038574, 0.9921875 ,
       0.1328125 , 0.99609375, 0.03735352, 0.03320312, 0.02929688,
       0.24511719, 0.09521484, 0.02294922, 0.20214844, 0.94140625,
       0.03735352, 0.06005859, 0.11914062, 0.11914062, 0.9609375 ,
       0.20214844, 0.9609375 , 0.01403809, 0.53125   , 0.03735352,
       0.9453125 , 0.9765625 , 0.1328125 , 0.06738281, 0.9921875 ,
       0.06005859, 0.99609375, 0.70703125, 0.03320312, 0.02929

In [12]:
y_true_arr

array([1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0,
       1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1,
       1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1,
       0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0,
       0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1,
       1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1,
       0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0,
       1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0,
       1, 1])